# Sistem Prediksi Kuat Tekan Beton

**Metode:** Multiple Linear Regression dan Random Forest Regression  
**Target:** Kuat tekan beton (MPa)  
**Dataset:** Concrete Compressive Strength — UCI Machine Learning Repository (DOI: 10.24432/C5PK67)

## Dasar Metode dan Metrik Evaluasi

### Multiple Linear Regression

**ŷ = β₀ + Σ⁸ⱼ₌₁ βⱼxⱼ**

ŷ menunjukkan prediksi kuat tekan. β₀ merupakan konstanta. βⱼ merupakan koefisien fitur ke-j. xⱼ merupakan nilai fitur ke-j.

### Random Forest Regression

**ŷ = (1/T) Σᵀₜ₌₁ fₜ(x)**

Setiap decision tree menghasilkan satu prediksi. Hasil akhir diperoleh dari rata-rata prediksi seluruh tree.

### Mean Absolute Error (MAE)

**MAE = (1/n) Σⁿᵢ₌₁ |yᵢ − ŷᵢ|**

MAE menunjukkan rata-rata selisih absolut antara nilai aktual dan nilai prediksi.

### Root Mean Squared Error (RMSE)

**RMSE = √[(1/n) Σⁿᵢ₌₁ (yᵢ − ŷᵢ)²]**

RMSE memberikan penalti lebih besar terhadap kesalahan prediksi yang tinggi.

### Coefficient of Determination (R²)

**R² = 1 − [Σⁿᵢ₌₁ (yᵢ − ŷᵢ)² / Σⁿᵢ₌₁ (yᵢ − ȳ)²]**

Nilai R² yang mendekati 1 menunjukkan kemampuan model yang lebih baik dalam menjelaskan variasi kuat tekan beton.

In [ ]:
!pip -q install ucimlrepo

import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ucimlrepo import fetch_ucirepo
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## Dataset

In [ ]:
concrete = fetch_ucirepo(id=165)
X = concrete.data.features.copy()
y = concrete.data.targets.copy()
data = pd.concat([X, y], axis=1)

FEATURES = list(X.columns)
TARGET = y.columns[0]

print('Ukuran data asli:', data.shape)
print('Jumlah nilai kosong:', int(data.isna().sum().sum()))
print('Jumlah duplikat identik:', int(data.duplicated().sum()))
display(data.head())
display(data.describe().T)

In [ ]:
data = data.drop_duplicates().reset_index(drop=True)
print('Ukuran setelah duplikat dihapus:', data.shape)

## Pembagian Data

Data dibagi menjadi 70% train, 15% validation, dan 15% test.

In [ ]:
train_validation, test = train_test_split(data, test_size=0.15, random_state=RANDOM_STATE)
train, validation = train_test_split(
    train_validation, test_size=0.15/0.85, random_state=RANDOM_STATE
)
print(f'Train: {len(train)}, validation: {len(validation)}, test: {len(test)}')

## Pemodelan dan Validasi

In [ ]:
def calculate_metrics(actual, predicted):
    return {
        'MAE (MPa)': mean_absolute_error(actual, predicted),
        'RMSE (MPa)': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
    }

models = {
    'Multiple Linear Regression': LinearRegression(),
    'Random Forest Regression': RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

validation_results = []
for name, model in models.items():
    model.fit(train[FEATURES], train[TARGET])
    predicted = model.predict(validation[FEATURES])
    validation_results.append({'Model': name, **calculate_metrics(validation[TARGET], predicted)})

validation_table = pd.DataFrame(validation_results)
display(validation_table.round(3))

In [ ]:
selected_name = validation_table.loc[validation_table['RMSE (MPa)'].idxmin(), 'Model']
print('Model terpilih:', selected_name)

validation_table.set_index('Model')[['MAE (MPa)', 'RMSE (MPa)']].plot(
    kind='bar', figsize=(8, 4), rot=0, color=['#2563eb', '#f59e0b']
)
plt.ylabel('MPa')
plt.title('Perbandingan Error pada Validation')
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Evaluasi pada Data Test

In [ ]:
train_validation = pd.concat([train, validation])
test_results = []
fitted_models = {}
test_predictions = {}

for name, model in models.items():
    fitted = clone(model)
    fitted.fit(train_validation[FEATURES], train_validation[TARGET])
    predicted = fitted.predict(test[FEATURES])
    fitted_models[name] = fitted
    test_predictions[name] = predicted
    test_results.append({'Model': name, **calculate_metrics(test[TARGET], predicted)})

test_table = pd.DataFrame(test_results)
display(test_table.round(3))

In [ ]:
best_model = fitted_models[selected_name]
best_prediction = test_predictions[selected_name]

lower = min(test[TARGET].min(), best_prediction.min())
upper = max(test[TARGET].max(), best_prediction.max())
plt.figure(figsize=(6, 5))
plt.scatter(test[TARGET], best_prediction, alpha=0.55)
plt.plot([lower, upper], [lower, upper], '--', color='red')
plt.xlabel('Aktual (MPa)')
plt.ylabel('Prediksi (MPa)')
plt.title('Aktual vs Prediksi pada Test')
plt.grid(alpha=0.25)
plt.show()

## Analisis Fitur dan Contoh Prediksi

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance_values = best_model.feature_importances_
else:
    importance_values = np.abs(best_model.coef_)

importance = pd.Series(importance_values, index=FEATURES).sort_values()
importance.plot(kind='barh', figsize=(8, 4), color='#16a34a')
plt.title('Pengaruh Relatif Fitur')
plt.tight_layout()
plt.show()

In [ ]:
example = pd.DataFrame([{
    'Cement': 265.0,
    'Blast Furnace Slag': 20.0,
    'Fly Ash': 0.0,
    'Water': 185.7,
    'Superplasticizer': 6.1,
    'Coarse Aggregate': 968.0,
    'Fine Aggregate': 780.0,
    'Age': 28,
}])[FEATURES]
example_prediction = best_model.predict(example)[0]
print(f'Prediksi contoh: {example_prediction:.2f} MPa')

## Penyimpanan Model Terbaik

In [ ]:
joblib.dump(best_model, 'best_model.joblib', compress=3)
print('Model tersimpan sebagai best_model.joblib')

try:
    from google.colab import files
    files.download('best_model.joblib')
except ImportError:
    pass

## Kesimpulan

Model terbaik ditentukan berdasarkan RMSE validation terendah, kemudian dievaluasi menggunakan data test. Hasil prediksi merupakan estimasi berbasis data historis dan tidak menggantikan pengujian kuat tekan di laboratorium.